# VLSP 2025 KD — Notebook 1: Teacher Distillation Only

**Goal**: Run teacher distillation (Qwen3.5-27B → reasoning traces), evaluate quality
**Output**: `distilled_sft.json` for NB3, match rate stats
**No student training** in this notebook — just measure what the teacher produces

| Step | Time (causal_conv1d) | Time (no causal_conv1d) |
|------|---------------------|------------------------|
| Test (50 samples) | ~5 min | ~14 min |
| Full (~3000 samples) | ~5-7h | ~13-15h ⚠️ |

> ⚠️ **WITHOUT causal_conv1d**: distillation may NOT finish in 12h.
> In that case, the notebook auto-saves checkpoints every 100 samples — start Session 2 and resume.

## Execution Order
1. **Run ALL cells with `TEST_MODE=True`** (~15 min) — verify no errors
2. Check Cell 7 stats (match rate should be >80%)
3. Set `TEST_MODE=False`, restart kernel, run all cells (full run, ~5-7h)
4. Download `distilled_sft.json` from outputs — use as input to NB3

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 1: CONFIGURATION — read this before running!
# ════════════════════════════════════════════════════════════════

# ▶ TEST BEFORE FULL RUN
TEST_MODE    = True     # ← SET TO False AFTER TEST PASSES
TEST_SAMPLES = 50       # samples in test mode (distillation only)

NOTEBOOK_ID = "distill-only"
OUTPUT_DIR  = f"/kaggle/working/outputs/{NOTEBOOK_ID}"

import os
from pathlib import Path
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print(f"{'='*60}")
print(f"Notebook  : {NOTEBOOK_ID}")
print(f"TEST_MODE : {TEST_MODE}  (n={TEST_SAMPLES if TEST_MODE else 'ALL'})")
print(f"Output    : {OUTPUT_DIR}")
print(f"{'='*60}")


In [ ]:
import subprocess, os, sys, shutil
from pathlib import Path

WORK_DIR     = Path("/kaggle/working/vlsp2025")
WHEELS_DIR   = Path("/kaggle/input/datasets/thanhduc1108/vlsp2025-kd-wheels")
PIPELINE_SRC = Path("/kaggle/input/datasets/thanhduc1108/vlsp2025-kd-pipeline")

os.environ.update({
    "HF_HUB_OFFLINE": "1",
    "TRANSFORMERS_OFFLINE": "1",
    "TOKENIZERS_PARALLELISM": "false",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
})
WORK_DIR.mkdir(parents=True, exist_ok=True)

# --- Install wheels ---
if WHEELS_DIR.exists():
    wheels = sorted(WHEELS_DIR.glob("*.whl"))
    if wheels:
        r = subprocess.run(
            [sys.executable, "-m", "pip", "install", "--quiet", "--no-index",
             f"--find-links={WHEELS_DIR}"] + [str(w) for w in wheels],
            capture_output=True, text=True)
        print(f"Wheels: {len(wheels)} installed" if r.returncode == 0 else f"Wheel warn: {r.stderr[:100]}")
else:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                    "transformers", "accelerate", "peft", "bitsandbytes", "sympy", "tqdm"],
                   check=False)
    print("Installed from PyPI (online mode)")

# --- Copy pipeline code (pipeline/, src/, configs/) ---
if PIPELINE_SRC.exists():
    for d in ["pipeline", "src", "configs"]:
        src_d = PIPELINE_SRC / d
        dst_d = WORK_DIR / d
        if src_d.exists():
            if dst_d.exists():
                shutil.rmtree(dst_d)
            shutil.copytree(src_d, dst_d)
            n = sum(1 for _ in dst_d.rglob("*") if _.is_file())
            print(f"  Copied {d}/ ({n} files)")
        else:
            print(f"  WARNING: {d}/ not found in pipeline dataset")
    # Verify critical module exists
    if not (WORK_DIR / "pipeline" / "__init__.py").exists():
        raise RuntimeError("pipeline/__init__.py missing after copy — re-upload vlsp2025-kd-pipeline dataset")
elif (WORK_DIR / "pipeline").exists():
    print(f"Pipeline already present: {WORK_DIR / 'pipeline'}")
else:
    raise RuntimeError(f"Pipeline not found at {PIPELINE_SRC}\nAdd dataset: thanhduc1108/vlsp2025-kd-pipeline")

if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))
os.chdir(WORK_DIR)
print(f"WORK_DIR: {WORK_DIR}")


In [ ]:
import sys, os, torch
from pathlib import Path

WORK_DIR = Path("/kaggle/working/vlsp2025")
if str(WORK_DIR) not in sys.path: sys.path.insert(0, str(WORK_DIR))
os.chdir(WORK_DIR)

if not torch.cuda.is_available():
    raise RuntimeError("No GPU! Enable RTX 6000 accelerator in notebook settings.")

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU:  {gpu_name}  ({vram_gb:.0f} GB)")

import transformers, peft, accelerate
print(f"Transformers: {transformers.__version__}  PEFT: {peft.__version__}")

try:
    import flash_attn
    HAS_FLASH = True
    print(f"Flash Attn: {flash_attn.__version__}")
except ImportError:
    HAS_FLASH = False
    print("Flash Attn: not available (sdpa fallback)")

GPU_PROFILE = "rtx6000_96gb" if vram_gb > 80 else ("a100_80gb" if vram_gb > 60 else "p100_16gb")
print(f"GPU Profile: {GPU_PROFILE}")

# Resolve offline model paths (Kaggle offline mode)
def resolve_model(hf_id):
    name = hf_id.split("/")[-1].lower()
    for root in [Path("/kaggle/input"), Path("/kaggle/models")]:
        if not root.exists(): continue
        for d in root.rglob("config.json"):
            parent = d.parent
            if name.replace("-","").replace("_","") in parent.name.lower().replace("-","").replace("_",""):
                print(f"  Found offline: {parent}")
                return str(parent)
    return hf_id

TEACHER_PATH = resolve_model("Qwen/Qwen3.5-27B")
STUDENT_PATH = resolve_model("Qwen/Qwen3.5-4B")
# Fallback to Kaggle model mount paths
if TEACHER_PATH == "Qwen/Qwen3.5-27B":
    p = Path("/kaggle/input/models/thanhduc1108/qwen_35_27b/transformers/default/1")
    if p.exists(): TEACHER_PATH = str(p)
if STUDENT_PATH == "Qwen/Qwen3.5-4B":
    p = Path("/kaggle/input/models/thanhduc1108/qwen_35_4b/transformers/default/1")
    if p.exists(): STUDENT_PATH = str(p)

print(f"Teacher: {TEACHER_PATH}")
print(f"Student: {STUDENT_PATH}")


In [ ]:
import sys, os
from pathlib import Path
from pipeline.config import load_config, save_config

WORK_DIR = Path("/kaggle/working/vlsp2025")
if str(WORK_DIR) not in sys.path: sys.path.insert(0, str(WORK_DIR))
os.chdir(WORK_DIR)

cfg = load_config(gpu_profile=GPU_PROFILE, overrides={
    "model": {"teacher_model": TEACHER_PATH, "student_model": STUDENT_PATH,
              "use_flash_attention": HAS_FLASH},
    "data": {
        "vinumqa_train":        "/kaggle/input/datasets/thanhduc1108/vinumericalqa-private/train.json",
        "vinumqa_valid":        "/kaggle/input/datasets/thanhduc1108/vinumericalqa-private/valid.json",
        "vinumqa_test":         "/kaggle/input/datasets/thanhduc1108/vinumericalqa-private/test.json",
        "vinumqa_private_test": "/kaggle/input/datasets/thanhduc1108/vinumericalqa-private/private_test.json",
        "finqa_dir":            "/kaggle/input/datasets/thanhduc1108/finqa-en",
        "max_samples":          TEST_SAMPLES if TEST_MODE else None,
    },
})
print(f"Teacher: {cfg.model.teacher_model.split('/')[-1]}")
print(f"batch_size={cfg.teacher.batch_size}  max_new_tokens={cfg.teacher.max_new_tokens}")
print(f"use_guided_template={cfg.teacher.use_guided_template}")
save_config(cfg, str(WORK_DIR / "data/pipeline/config_distill.yaml"))


In [ ]:
import os, sys, time, json
from pathlib import Path
from pipeline.data_prep import run_data_prep

WORK_DIR = Path("/kaggle/working/vlsp2025")
if str(WORK_DIR) not in sys.path: sys.path.insert(0, str(WORK_DIR))
os.chdir(WORK_DIR)

print("=" * 60)
print("DATA PREPARATION")
print("=" * 60)
t0 = time.time()
data_paths = run_data_prep(cfg)
print(f"\nCompleted in {time.time()-t0:.1f}s")

# Print sample counts
for k, v in data_paths.items():
    try:
        d = json.load(open(v))
        print(f"  {k}: {len(d)} samples  ({v})")
    except Exception:
        print(f"  {k}: {v}")


In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 6: TIMING ESTIMATE — check before committing to full run
# ════════════════════════════════════════════════════════════════
import json, torch
from pathlib import Path

WORK_DIR     = Path("/kaggle/working/vlsp2025")
pipeline_out = WORK_DIR / "data/pipeline"

total = done = 0
if (pipeline_out / "teacher_input.json").exists():
    dataset = json.load(open(pipeline_out / "teacher_input.json"))
    total = len(dataset)
if (pipeline_out / "teacher_raw_checkpoint.json").exists():
    ckpt = json.load(open(pipeline_out / "teacher_raw_checkpoint.json"))
    done = len(ckpt)
    matched = sum(1 for r in ckpt if r.get("matched", False))
    print(f"Checkpoint: {done}/{total} done, {matched} matched ({matched/max(done,1)*100:.0f}%)")

remaining = total - done
try:
    import causal_conv1d
    tps = 40  # tokens/sec
    conv1d_status = "INSTALLED (fast)"
except ImportError:
    tps = 17  # tokens/sec
    conv1d_status = "MISSING (2.3x slower — risk timeout!)"

batch = cfg.teacher.batch_size
avg_out = 280  # guided template output avg tokens
s_per_sample = avg_out / tps  # seconds per sample (batch parallelism)
eta_h = remaining * s_per_sample / 3600

print(f"\ncausal_conv1d : {conv1d_status}")
print(f"batch_size    : {batch}  |  max_new_tokens: {cfg.teacher.max_new_tokens}")
print(f"Throughput est: {tps} tok/s  =>  {s_per_sample:.1f}s/sample")
print(f"Remaining     : {remaining} samples")
print(f"Est. time     : {eta_h:.1f}h")
if eta_h > 10.5:
    print(f"\nWARNING: May exceed 12h! Consider:")
    print(f"  - Reducing total samples (cfg.data.max_samples)")
    print(f"  - Using 2-session approach (checkpoint auto-saves every 100 samples)")


In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 7: TEACHER DISTILLATION
# Checkpoints every 100 samples — safe to interrupt and resume
# ════════════════════════════════════════════════════════════════
import gc, os, sys, time, torch
from pathlib import Path
from pipeline.teacher_distill import run_teacher_distillation

WORK_DIR = Path("/kaggle/working/vlsp2025")
os.chdir(WORK_DIR)

print("=" * 65)
print("TEACHER DISTILLATION")
print(f"  guided_template : {cfg.teacher.use_guided_template}")
print(f"  max_new_tokens  : {cfg.teacher.max_new_tokens}")
print(f"  batch_size      : {cfg.teacher.batch_size}")
print(f"  checkpoint_every: {cfg.teacher.checkpoint_every}")
print("=" * 65)

t0 = time.time()
distilled_path = run_teacher_distillation(
    cfg,
    teacher_input_path=data_paths["teacher_input"],
    resume=True,   # safe to restart — picks up from checkpoint
)
elapsed = time.time() - t0
print(f"\nDistillation done in {elapsed/3600:.2f}h  ({elapsed:.0f}s)")
print(f"Output: {distilled_path}")

# ── Aggressive GPU cleanup ──────────────────────────────────────
for _n in list(globals()):
    if "teacher" in _n.lower():
        try: del globals()[_n]
        except: pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache(); torch.cuda.synchronize()
    free_b, total_b = torch.cuda.mem_get_info()
    print(f"GPU free after cleanup: {free_b/1024**3:.1f} / {total_b/1024**3:.1f} GB")


In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 8: EVALUATE DISTILLATION QUALITY
# Shows how well teacher traces match gold answers
# ════════════════════════════════════════════════════════════════
import json
from pathlib import Path

WORK_DIR = Path("/kaggle/working/vlsp2025")
pipeline_out = WORK_DIR / "data/pipeline"

# Load distilled data
with open(distilled_path, "r", encoding="utf-8") as f:
    distilled = json.load(f)

# Load raw checkpoint for full stats
ckpt_path = pipeline_out / "teacher_raw_checkpoint.json"
raw_stats = {"exact_match": 0, "answer_match": 0, "program_valid": 0, "invalid": 0, "error": 0}
total_raw = 0
if ckpt_path.exists():
    raw = json.load(open(ckpt_path))
    total_raw = len(raw)
    for r in raw:
        mt = r.get("match_type", "invalid")
        raw_stats[mt] = raw_stats.get(mt, 0) + 1

print(f"\n{'='*65}")
print(f"DISTILLATION QUALITY REPORT")
print(f"{'='*65}")
print(f"Total processed    : {total_raw}")
print(f"Distilled (usable) : {len(distilled)}  ({len(distilled)/max(total_raw,1)*100:.1f}%)")
print()
print(f"{'Match type':<25} {'Count':>7}  {'%':>6}")
print(f"{'-'*42}")
for mt, cnt in sorted(raw_stats.items(), key=lambda x: -x[1]):
    pct = cnt / max(total_raw, 1) * 100
    bar = '█' * int(pct / 3)
    print(f"  {mt:<23} {cnt:>7}  {pct:>5.1f}%  {bar}")
print(f"{'='*65}")
print(f"\nQuality breakdown:")
usable = raw_stats.get("exact_match", 0) + raw_stats.get("answer_match", 0)
print(f"  Usable for SFT (matched): {usable}/{total_raw} = {usable/max(total_raw,1)*100:.1f}%")
print(f"  Avg output length: ~280 tokens (guided template)")
print(f"\nData saved to: {distilled_path}")


In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 9: SAVE OUTPUTS
# Saves distilled_sft.json for use in NB3
# ════════════════════════════════════════════════════════════════
import json, shutil
from pathlib import Path

WORK_DIR = Path("/kaggle/working/vlsp2025")
out = Path(OUTPUT_DIR)
out.mkdir(parents=True, exist_ok=True)
pipeline_out = WORK_DIR / "data/pipeline"

# Copy files to /kaggle/working/outputs/distill-only/
FILES_TO_SAVE = [
    "distilled_sft.json",       # ← import this in NB3!
    "teacher_raw_checkpoint.json",
    "teacher_raw_output.json",
    "sft_train.json",
    "sft_valid.json",
    "config_distill.yaml",
]
for fname in FILES_TO_SAVE:
    src = pipeline_out / fname
    if src.exists():
        shutil.copy2(src, out / fname)
        sz = src.stat().st_size / 1024
        print(f"  Saved: {fname}  ({sz:.0f} KB)")
    else:
        print(f"  (skip) {fname} not found")

# Summary
import json as _json
raw_ckpt_path = pipeline_out / "teacher_raw_checkpoint.json"
if raw_ckpt_path.exists():
    raw = _json.load(open(raw_ckpt_path))
    matched = sum(1 for r in raw if r.get("matched", False))
    summary = {
        "notebook": NOTEBOOK_ID,
        "test_mode": TEST_MODE,
        "total_processed": len(raw),
        "matched": matched,
        "match_rate": matched / max(len(raw), 1),
        "distilled_path": str(out / "distilled_sft.json"),
    }
    with open(out / "distill_summary.json", "w", encoding="utf-8") as f:
        _json.dump(summary, f, ensure_ascii=False, indent=2)
    print(f"\n  Summary: {matched}/{len(raw)} matched ({matched/max(len(raw),1)*100:.1f}%)")

print(f"\nAll outputs -> {OUTPUT_DIR}")
print(f"Files: {[p.name for p in sorted(out.iterdir())]}")
print("\nNEXT STEP: Use distilled_sft.json in NB3 (rtx6000-eval-kd.ipynb)")
